# Pilot Analysis - Context-Aware Prompting (N=30)

**Model:** gpt-5.4-mini-2026-03-17  
**Method:** Context-Aware Few-shot, Temperature=0  
**Pilot sample:** 10 per domain (E-commerce, Finance/Banking, Social Network)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import os

sns.set_theme(style='whitegrid')
figures_dir = '../figures'
os.makedirs(figures_dir, exist_ok=True)

## 1. Load Data

In [ ]:
df = pd.read_csv('pilot_summary.csv')
print(f'Total samples: {len(df)}')
display(df.head())

## 2. Descriptive Statistics

In [ ]:
display(df[['raw_cosine', 'skeleton_cosine']].describe())
print('\nExecutable counts:')
display(df['executable'].value_counts())

## 3. Figure 1: Distribution of Skeleton Cosine (Pilot)

Histogram + Boxplot combo

In [ ]:
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(10, 6),
                                       sharex=True,
                                       gridspec_kw={'height_ratios': [1, 4], 'hspace': 0.05})

sns.boxplot(x=df['skeleton_cosine'], color='mediumpurple', ax=ax_box)
ax_box.axvline(0.85, color='red', linestyle='--', linewidth=2)
ax_box.set_yticks([])
ax_box.set_title('Distribution of Skeleton Cosine (Pilot, N=30)', fontsize=14)

ax_hist.hist(df['skeleton_cosine'], bins=10, color='mediumpurple', edgecolor='black', alpha=0.8)
ax_hist.axvline(0.85, color='red', linestyle='--', linewidth=2, label='Threshold = 0.85')
median_val = df['skeleton_cosine'].median()
ax_hist.axvline(median_val, color='orange', linestyle='-', linewidth=2, label=f'Median = {median_val:.4f}')
ax_hist.set_xlabel('Skeleton Cosine Similarity', fontsize=12)
ax_hist.set_ylabel('Count', fontsize=12)
ax_hist.text(0.02, 0.92, f'N = {len(df)}', transform=ax_hist.transAxes, fontsize=12,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax_hist.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'pilot_fig1_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/pilot_fig1_distribution.png')

## 4. Figure 2: Executable Rate (Pilot)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
exec_counts = df['executable'].value_counts()
bars = ax.bar(exec_counts.index, exec_counts.values, color=['#66b3ff', '#ff9999'], edgecolor='black', width=0.5)
ax.axhline(y=len(df)*0.80, color='red', linestyle='--', linewidth=2, label=f'Threshold 80% ({int(len(df)*0.80)} samples)')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5, f'{int(height)}\n({height/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Executable Rate (Pilot, N=30)', fontsize=14)
ax.set_xlabel('Executable Status (behave --dry-run)')
ax.set_ylabel('Number of User Stories')
ax.text(0.02, 0.92, f'N = {len(df)}', transform=ax.transAxes, fontsize=12,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.legend(loc='upper right')
ax.set_ylim(0, max(exec_counts.values) + 5)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'pilot_fig2_executable.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/pilot_fig2_executable.png')

## 5. Summary Table

In [ ]:
print('='*50)
print('       PILOT DESCRIPTIVE SUMMARY')
print('='*50)
for col in ['raw_cosine', 'skeleton_cosine']:
    print(f'\n--- {col} ---')
    print(f'  Mean:   {df[col].mean():.4f}')
    print(f'  Median: {df[col].median():.4f}')
    print(f'  Std:    {df[col].std():.4f}')
    print(f'  Min:    {df[col].min():.4f}')
    print(f'  Max:    {df[col].max():.4f}')
pass_count = (df['executable'] == 'PASS').sum()
print(f'\n--- Executable Rate ---')
print(f'  PASS: {pass_count}/{len(df)} ({pass_count/len(df)*100:.1f}%)')
print('\nNote: No hypothesis tests for pilot (N=30 is too small for publication-grade inference).')

## 6. API Cost Analysis

In [ ]:
logs = []
with open('api_logs_pilot.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        logs.append(json.loads(line))

df_logs = pd.DataFrame(logs)
print('='*50)
print('       API COST ANALYSIS (PILOT)')
print('='*50)
print(f'  Total API calls:       {len(df_logs)}')
print(f'  Total prompt tokens:   {df_logs["prompt_tokens"].sum():,}')
print(f'  Total completion tokens:{df_logs["completion_tokens"].sum():,}')
print(f'  Total tokens:          {df_logs["total_tokens"].sum():,}')
print(f'  Total cost (USD):      ${df_logs["cost_usd"].sum():.4f}')
print(f'  Avg cost per call:     ${df_logs["cost_usd"].mean():.6f}')